Implement HITL for a food ordering application.

HITL Points

-   Confirm restuarant
-   Confirm total price
-   Confirm details before placing order like quantity, address and total price
-   Cancel order

In [73]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

In [74]:
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")

In [75]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [76]:
from langchain.agents.structured_output import ProviderStrategy

In [77]:
restaurant_list = [
    {
        "name": "Royal Rajasthan",
        "cuisine": "Rajasthani",
        "location": "Downtown",
        "food_items": [
            {
                "name": "Dal Baati Churma",
                "category": "Main Course",
                "price": 320
            },
            {
                "name": "Gatte Ki Sabzi",
                "category": "Main Course",
                "price": 240
            },
            {
                "name": "Ker Sangri",
                "category": "Vegetarian",
                "price": 220
            },
            {
                "name": "Pyaaz Kachori",
                "category": "Starter",
                "price": 100
            },
            {
                "name": "Ghevar",
                "category": "Dessert",
                "price": 150
            }
        ]
    },

    {
        "name": "Coastal Spice",
        "cuisine": "South Indian",
        "location": "Green Park",
        "food_items": [
            {
                "name": "Masala Dosa",
                "category": "Main Course",
                "price": 180
            },
            {
                "name": "Idli Sambar",
                "category": "Breakfast",
                "price": 120
            },
            {
                "name": "Medu Vada",
                "category": "Starter",
                "price": 110
            },
            {
                "name": "Uttapam",
                "category": "Main Course",
                "price": 160
            },
            {
                "name": "Payasam",
                "category": "Dessert",
                "price": 130
            }
        ]
    },

    {
        "name": "Punjab House",
        "cuisine": "Punjabi",
        "location": "Connaught Place",
        "food_items": [
            {
                "name": "Amritsari Kulcha",
                "category": "Main Course",
                "price": 220
            },
            {
                "name": "Chole Bhature",
                "category": "Main Course",
                "price": 200
            },
            {
                "name": "Sarson Da Saag",
                "category": "Vegetarian",
                "price": 260
            },
            {
                "name": "Tandoori Chicken",
                "category": "Starter",
                "price": 350
            },
            {
                "name": "Phirni",
                "category": "Dessert",
                "price": 140
            }
        ]
    },
    {
    "name": "Curry Corner",
    "cuisine": "Indian",
    "location": "Midtown",
    "food_items": [
        {
            "name": "Butter Chicken",
            "category": "Main Course",
            "price": 350,
            "available": True
        },
        {
            "name": "Paneer Tikka",
            "category": "Starter",
            "price": 280,
            "available": True
        },
        {
            "name": "Biryani",
            "category": "Main Course",
            "price": 300,
            "available": True
        },
        {
            "name": "Dal Makhani",
            "category": "Main Course",
            "price": 220,
            "available": True
        },
        {
            "name": "Naan",
            "category": "Bread",
            "price": 60,
            "available": True
        }
    ]
}
]

In [78]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key,
    temperature=0
)

In [79]:
class RequestedFoodAndRestaurant(BaseModel):
    food_item: str | None = Field(
        default=None,
        description="Contains the name of the food item the user is asking for."
    )

    restaurant: str | None = Field(
        default=None,
        description="Contains the name of the restaurant the user is asking for."
    )

In [80]:
class RequestedFoodAndRestaurantCatelog(BaseModel):
    catelog_item:list[RequestedFoodAndRestaurant] = Field(..., description="This contains the object of a RequestedFoodAndRestaurant")

In [81]:
class OrderCatelogDetails(BaseModel):
    resturant: str = Field(..., description="Name of the restaurant")
    localtion: str = Field(..., description="Location of the restaurant")
    food_item: str = Field(..., description="Name of the food item")
    price: float = Field(..., description="Price of the food item")

In [82]:
food_and_restaurant_identification_agent = create_agent(
    model= llm,
    response_format=ProviderStrategy(RequestedFoodAndRestaurantCatelog),
    system_prompt="You are a food and restaurant identification agent. Your task is to identify the food items and restaurants from the user's query"
)

In [83]:
response = food_and_restaurant_identification_agent.invoke({
    "messages":[("user", "I want to order Butter Chicken and Biryani")]
})
print(response)
print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='I want to order Butter Chicken and Biryani',
            additional_kwargs={},
            response_metadata={},
            id='5d114c19-dad9-4f56-bc89-b9e47741ab86'
        ),
        AIMessage(
            content='{"catelog_item":[{"food_item":"Butter 
Chicken","restaurant":null},{"food_item":"Biryani","restaurant":null}]}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 37,
                    'prompt_tokens': 213,
                    'total_tokens': 250,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-4.1-mini-2025-04-14',
                'system_fingerprint': 'fp_a4a7716a60',
                'id': 'chatcmpl-EMfBZrlrs8ZNeq6jeqY6DlkPafzKs',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a08ce5-6b07-7503-a711-cd2a2a6c2ca2-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 213,
                'output_tokens': 37,
                'total_tokens': 250,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': RequestedFoodAndRestaurantCatelog(
        catelog_item=[
            RequestedFoodAndRestaurant(food_item='Butter Chicken', restaurant=None),
            RequestedFoodAndRestaurant(food_item='Biryani', restaurant=None)
        ]
    )
}

{"catelog_item":[{"food_item":"Butter Chicken","restaurant":null},{"food_item":"Biryani","restaurant":null}]}

In [84]:
query = "I want to order Butter Chicken and Biryani from Curry Corner"

In [85]:
requested_catelog = []

In [90]:

def search_restaurant_for_the_food_item(query: str):
    """
    Search for restaurants that have the requested food item.
    """
    restaurant_catalog = globals().get("restaurant_list", [])
    if not restaurant_catalog:
        raise NameError(
            "restaurant_list is not defined. Run the restaurant catalog cell before calling this function."
        )

    result = food_and_restaurant_identification_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    catalog = result["structured_response"]

    search_results = []

    for item in catalog.catelog_item:

        food_item = item.food_item
        restaurant_name = item.restaurant

        for restaurant in restaurant_catalog:

            # Food item only
            if food_item is not None and restaurant_name is None:

                for menu_item in restaurant["food_items"]:

                    if menu_item["name"].lower() == food_item.lower():

                        search_results.append({
                            "restaurant": restaurant["name"],
                            "food_item": menu_item["name"],
                            "price": menu_item["price"],
                            "location": restaurant["location"]
                        })

            # Restaurant only
            elif restaurant_name is not None and food_item is None:

                if restaurant["name"].lower() == restaurant_name.lower():

                        search_results.append({
                            "restaurant": restaurant["name"],
                            "food_item": menu_item["name"],
                            "price": menu_item["price"],
                            "location": restaurant["location"]
                        })

            # Both restaurant and food item
            elif restaurant_name is not None and food_item is not None:

                if restaurant["name"].lower() == restaurant_name.lower():

                    for menu_item in restaurant["food_items"]:

                        if menu_item["name"].lower() == food_item.lower():

                            search_results.append({
                                "restaurant": restaurant["name"],
                                "food_item": menu_item["name"],
                                "price": menu_item["price"],
                                "location": restaurant["location"]
                            })

    return search_results

In [91]:
e = search_restaurant_for_the_food_item("I want to order Butter Chicken and Biryani from Curry Corner")
print(e)

[
    {'restaurant': 'Curry Corner', 'food_item': 'Butter Chicken', 'price': 350, 'location': 'Midtown'},
    {'restaurant': 'Curry Corner', 'food_item': 'Biryani', 'price': 300, 'location': 'Midtown'}
]

In [ ]:
def get_user_order_exclusively(query:str):
    """
        Get the order explicitely provided by the user.
    """
   

In [ ]:
def confirm_food_item_and_restraunt():
    pass

In [ ]:
def create_order():
    pass

In [ ]:
def get_order_details():
    pass

In [ ]:
def cancel_order():
    pass

In [45]:
system_prompt = """
            You are a helpful agent whose job is to analyse the user query
            and search the appropriate restuarant and food item user is asking for.
            You have access of different tools for searching the food and resturant,
            creating the order, fetching the order details and canceling the order.
"""

In [ ]:
agent = create_agent(
    model = llm,
    system_prompt= system_prompt
)